## Section 0: Configuration & Constants

In [ ]:
RANDOM_SEED = 231

NUM_CLASSES = 4
PARTICLE_NAMES = ['Pion', 'Kaon', 'Proton', 'Electron']
PDG_TO_SPECIES = {
    211: 0,
    321: 1,
    2212: 2,
    11: 3,
}

MOMENTUM_RANGES = [
    {'key': 'full', 'name': 'Full Spectrum (0.1-∞ GeV/c)', 'min': 0.1, 'max': float('inf')},
    {'key': '0.7-1.5', 'name': '0.7-1.5 GeV/c', 'min': 0.7, 'max': 1.5},
    {'key': '1-3', 'name': '1-3 GeV/c', 'min': 1.0, 'max': 3.0},
]

CSV_PATTERN = '/kaggle/input/datasets/robertforynski/new-ao2d-lhc25f60544122/pid_features_*.csv'

TRAINING_FEATURES = [
    'pt', 'eta', 'phi',
    'tpc_signal', 'tpc_nsigma_pi', 'tpc_nsigma_ka', 'tpc_nsigma_pr', 'tpc_nsigma_el',
    'tof_beta', 'tof_nsigma_pi', 'tof_nsigma_ka', 'tof_nsigma_pr', 'tof_nsigma_el',
    'bayes_prob_pi', 'bayes_prob_ka', 'bayes_prob_pr', 'bayes_prob_el',
    'dca_xy', 'dca_z',
    'has_tpc', 'has_tof',
]

DETECTOR_GROUPS = {
    'tpc': ['tpc_signal', 'tpc_nsigma_pi', 'tpc_nsigma_ka', 'tpc_nsigma_pr', 'tpc_nsigma_el'],
    'tof': ['tof_beta', 'tof_nsigma_pi', 'tof_nsigma_ka', 'tof_nsigma_pr', 'tof_nsigma_el'],
    'bayes': ['bayes_prob_pi', 'bayes_prob_ka', 'bayes_prob_pr', 'bayes_prob_el'],
    'kinematics': ['pt', 'eta', 'phi', 'dca_xy', 'dca_z'],
}

MODEL_TYPES = [
    'JAX_SimpleNN',
    'JAX_DNN',
    'JAX_FSE_Attention',
    'JAX_FSE_Attention_DetectorAware',
]

HYPERPARAMETERS = {
    'JAX_SimpleNN': {
        'hidden_dims': [512, 256, 128, 64],
        'dropout_rate': 0.5,
        'learning_rate': 1e-4,
        'batch_size': 256,
        'num_epochs': 100,
        'patience': 30,
    },
    'JAX_DNN': {
        'hidden_dims': [1024, 512, 256, 128, 64],
        'dropout_rate': 0.5,
        'learning_rate': 5e-5,
        'batch_size': 256,
        'num_epochs': 100,
        'patience': 30,
    },
    'JAX_FSE_Attention': {
        'hidden_dim': 64,
        'num_heads': 4,
        'dropout_rate': 0.5,
        'learning_rate': 1e-4,
        'batch_size': 256,
        'num_epochs': 100,
        'patience': 30,
    },
    'JAX_FSE_Attention_DetectorAware': {
        'hidden_dim': 64,
        'num_heads': 4,
        'dropout_rate': 0.5,
        'learning_rate': 1e-4,
        'batch_size': 256,
        'num_epochs': 100,
        'patience': 30,
        'detector_embed_dim': 8,
    },
}

FORCE_TRAINING = {
    'JAX_SimpleNN': {'full': False, '0.7-1.5': False, '1-3': False},
    'JAX_DNN': {'full': False, '0.7-1.5': False, '1-3': False},
    'JAX_FSE_Attention': {'full': False, '0.7-1.5': False, '1-3': False},
    'JAX_FSE_Attention_DetectorAware': {'full': False, '0.7-1.5': False, '1-3': False},
}

model_colors = {
    'JAX_SimpleNN': '#3B82F6',           # blue
    'JAX_DNN': '#F59E0B',               # amber
    'JAX_FSE_Attention': '#22C55E',     # green
    'JAX_FSE_Attention_DetectorAware': '#EF4444',  # red
    'Bayesian_PID': '#F97316',          # orange
}

print("Config ready.")


## Section 1: Imports

In [ ]:
import os
import glob
import time
import pickle
import warnings

import gc

import numpy as np
import pandas as pd

import jax
import jax.numpy as jnp
from jax import random, jit

import optax
from flax import linen as nn
from flax.training import train_state

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix, accuracy_score
from sklearn.utils.class_weight import compute_class_weight

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import (
    roc_curve,
    auc,
    precision_recall_curve,
    classification_report,
)
from sklearn.preprocessing import label_binarize

warnings.filterwarnings("ignore")

print("JAX version:", jax.__version__)
print("Devices:", jax.devices())


## Section 2: Data Loading & Preprocessing

In [ ]:
def pdg_to_species(pdg):
    if pd.isna(pdg):
        return -1
    ap = abs(int(pdg))
    return PDG_TO_SPECIES.get(ap, -1)


def load_all_csv(CSV_PATTERN):

    files = sorted(glob.glob(CSV_PATTERN))
    if not files:
        raise FileNotFoundError(f"No CSV files found for pattern: {CSV_PATTERN}")

    dfs = []

    for f in files:
        print("Loading:", os.path.basename(f))

        # Load CSV with robust na handling
        df_file = pd.read_csv(
            f,
            na_values=['-', 'nan', 'null', 'NaN', 'NULL', ''],
            keep_default_na=True,
            on_bad_lines='skip',
            low_memory=False
        )

        dfs.append(df_file)

    # Combine
    df = pd.concat(dfs, ignore_index=True)
    print(f"Loaded {len(df):,} rows from {len(files)} files.")

    # Safe numeric conversion
    for col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='ignore')
        if pd.api.types.is_float_dtype(df[col]):
            df[col] = df[col].astype('float32')
        if col == 'mc_pdg':
            df[col] = pd.to_numeric(df[col], errors='coerce').astype('Int32')

    gc.collect()
    return df


def preprocess_momentum_range(df, momentum_range):

    print("\nPreprocessing:", momentum_range['name'])

    # 1) Momentum filter
    df_f = df[(df['p'] >= momentum_range['min']) &
              (df['p'] < momentum_range['max'])].copy()
    print(f"  Tracks in range: {len(df_f):,}")

    # 2) Clean MC truth
    y_raw = pd.to_numeric(df_f['mc_pdg'], errors='coerce')
    valid_truth = y_raw.notna()
    df_f = df_f[valid_truth].copy()
    y_raw = y_raw[valid_truth]
    print(f"  After MC truth cleaning: {len(df_f):,}")

    # 3) Bayes mask BEFORE filling
    bayes_features = DETECTOR_GROUPS['bayes']
    bayes_complete_before = df_f[bayes_features].notnull().all(axis=1)
    bayes_pred_before = np.argmax(df_f[bayes_features].fillna(-1).values, axis=1)

    # 4) Fill detector features
    for feat in DETECTOR_GROUPS['tof']:
        if feat in df_f.columns:
            df_f[feat] = df_f[feat].fillna(0.0 if feat == 'tof_beta' else 999.0)

    for feat in DETECTOR_GROUPS['tpc']:
        if feat in df_f.columns:
            df_f[feat] = df_f[feat].fillna(0.0 if feat == 'tpc_signal' else 999.0)

    for feat in bayes_features:
        if feat in df_f.columns:
            df_f[feat] = df_f[feat].fillna(0.25)

    for feat in DETECTOR_GROUPS['kinematics']:
        if feat in df_f.columns:
            df_f[feat] = df_f[feat].fillna(df_f[feat].median())

    df_f['has_tpc'] = df_f['has_tpc'].fillna(0)
    df_f['has_tof'] = df_f['has_tof'].fillna(0)

    df_f = df_f.dropna(subset=TRAINING_FEATURES)

    # 5) Map PDG → species
    y = np.array([pdg_to_species(v) for v in y_raw])
    valid_species = y >= 0
    df_f = df_f.iloc[valid_species].copy()
    y = y[valid_species]
    bayes_complete_before = bayes_complete_before.iloc[valid_species]
    bayes_pred_before = bayes_pred_before[valid_species]

    # 6) Feature matrix → float32 numpy
    X = df_f[TRAINING_FEATURES].to_numpy(dtype=np.float32, copy=False)

    # Free pandas memory early
    del df_f
    gc.collect()

    # Group masks
    group_names = list(DETECTOR_GROUPS.keys())
    masks = []
    for g in group_names:
        if g == 'tpc':
            masks.append(bayes_complete_before.values.astype('float32'))  # example
        elif g == 'tof':
            masks.append(bayes_complete_before.values.astype('float32'))  # example
        else:
            masks.append(np.ones(len(X), dtype='float32'))
    group_masks = np.stack(masks, axis=1)

    bayes_mask = bayes_complete_before.values.astype('float32')
    bayes_pred = bayes_pred_before

    print("  Final shape:", X.shape)
    for i, name in enumerate(PARTICLE_NAMES):
        n = np.sum(y == i)
        print(f"    {name:10s}: {n:7d} ({n/len(y):5.2%})")

    # 7) Detector modes
    has_tpc = X[:, TRAINING_FEATURES.index('has_tpc')].astype(int)
    has_tof = X[:, TRAINING_FEATURES.index('has_tof')].astype(int)
    detector_mode = np.zeros(len(X), dtype='int32')
    detector_mode[(has_tpc == 1) & (has_tof == 0)] = 1
    detector_mode[(has_tpc == 0) & (has_tof == 1)] = 2
    detector_mode[(has_tpc == 1) & (has_tof == 1)] = 3

    # 8) Train/test split (still safe)
    X_train, X_test, y_train, y_test, masks_train, masks_test, \
    bayes_mask_train, bayes_mask_test, bayes_pred_train, bayes_pred_test, \
    modes_train, modes_test = train_test_split(
        X,
        y,
        group_masks,
        bayes_mask,
        bayes_pred,
        detector_mode,
        test_size=0.2,
        random_state=RANDOM_SEED,
        stratify=y,
    )

    # Free full arrays
    del X, y, group_masks, bayes_mask, bayes_pred, detector_mode
    gc.collect()

    # 9) StandardScaler: fit on training only, transform on batches later
    scaler = StandardScaler()
    scaler.fit(X_train)

    print(f"  Train: {len(X_train):,}, Test: {len(X_test):,}")

    return {
        'X_train': X_train,
        'X_test': X_test,
        'y_train': y_train,
        'y_test': y_test,
        'scaler': scaler,
        'features': TRAINING_FEATURES,
        'masks_train': masks_train,
        'masks_test': masks_test,
        'group_names': group_names,
        'bayes_availability_test': bayes_mask_test,
        'bayes_pred_original_test': bayes_pred_test,
        'detector_modes_train': modes_train,
        'detector_modes_test': modes_test,
    }
    

## Section 3: Model Definitions & Training

In [ ]:
def focal_loss(logits, labels, class_weights=None, alpha=0.25, gamma=2.0):
    probs = jax.nn.softmax(logits, axis=-1)
    pt = probs[jnp.arange(labels.shape[0]), labels]
    ce = -jnp.log(pt + 1e-7)
    w = class_weights[labels] if class_weights is not None else 1.0
    fl = alpha * (1.0 - pt) ** gamma
    return jnp.mean(w * fl * ce)


class JAX_SimpleNN(nn.Module):
    hidden_dims: list
    num_classes: int
    dropout_rate: float = 0.3

    @nn.compact
    def __call__(self, x, training: bool = False):
        for d in self.hidden_dims:
            x = nn.Dense(d)(x)
            x = nn.relu(x)
            x = nn.Dropout(rate=self.dropout_rate, deterministic=not training)(x)
        return nn.Dense(self.num_classes)(x)


class JAX_DNN(nn.Module):
    hidden_dims: list
    num_classes: int
    dropout_rate: float = 0.3

    @nn.compact
    def __call__(self, x, training: bool = False):
        for d in self.hidden_dims:
            x = nn.Dense(d)(x)
            x = nn.BatchNorm(use_running_average=not training)(x)
            x = nn.relu(x)
            x = nn.Dropout(rate=self.dropout_rate, deterministic=not training)(x)
        return nn.Dense(self.num_classes)(x)


class JAX_FSE_Attention(nn.Module):
    hidden_dim: int = 64
    num_heads: int = 4
    num_classes: int = 4
    dropout_rate: float = 0.3

    @nn.compact
    def __call__(self, x, group_mask, training: bool = False):
        b = x.shape[0]
        g = group_mask.shape[1]

        h = self.hidden_dim
        proj = nn.Dense(h * g)(x).reshape(b, g, h)
        proj = proj * group_mask[:, :, None]

        attn_mask = group_mask[:, None, None, :]
        attn = nn.MultiHeadDotProductAttention(num_heads=self.num_heads)(proj, proj, mask=attn_mask)
        attn = nn.LayerNorm()(attn)

        gates = nn.Dense(h)(attn)
        gates = nn.sigmoid(gates)
        gated = attn * gates

        denom = jnp.clip(jnp.sum(group_mask, axis=1, keepdims=True), a_min=1.0)
        pooled = jnp.sum(gated * group_mask[:, :, None], axis=1) / denom

        x = nn.Dense(128)(pooled)
        x = nn.relu(x)
        x = nn.Dropout(rate=self.dropout_rate, deterministic=not training)(x)
        x = nn.Dense(64)(x)
        x = nn.relu(x)
        x = nn.Dropout(rate=self.dropout_rate, deterministic=not training)(x)
        return nn.Dense(self.num_classes)(x)


class JAX_FSE_Attention_DetectorAware(nn.Module):
    hidden_dim: int = 64
    num_heads: int = 4
    num_classes: int = 4
    dropout_rate: float = 0.3
    detector_embed_dim: int = 8

    @nn.compact
    def __call__(self, x, group_mask, detector_mode, training: bool = False):
        b = x.shape[0]
        g = group_mask.shape[1]
        h = self.hidden_dim

        proj = nn.Dense(h * g)(x).reshape(b, g, h)
        proj = proj * group_mask[:, :, None]

        attn_mask = group_mask[:, None, None, :]
        attn = nn.MultiHeadDotProductAttention(num_heads=self.num_heads)(proj, proj, mask=attn_mask)
        attn = nn.LayerNorm()(attn)

        gates = nn.Dense(h)(attn)
        gates = nn.sigmoid(gates)
        gated = attn * gates

        denom = jnp.clip(jnp.sum(group_mask, axis=1, keepdims=True), a_min=1.0)
        pooled = jnp.sum(gated * group_mask[:, :, None], axis=1) / denom

        onehot = jax.nn.one_hot(detector_mode, num_classes=4)
        det_emb = nn.Dense(self.detector_embed_dim)(onehot)
        det_emb = nn.relu(det_emb)

        fused = jnp.concatenate([pooled, det_emb], axis=-1)

        x = nn.Dense(128)(fused)
        x = nn.relu(x)
        x = nn.Dropout(rate=self.dropout_rate, deterministic=not training)(x)
        x = nn.Dense(64)(x)
        x = nn.relu(x)
        x = nn.Dropout(rate=self.dropout_rate, deterministic=not training)(x)
        return nn.Dense(self.num_classes)(x)


class TrainStateWithBatchStats(train_state.TrainState):
    batch_stats: any = None


@jit
def train_step_simplenn(state, x, y, rng, class_weights):
    def loss_fn(params):
        logits = state.apply_fn({'params': params}, x, training=True, rngs={'dropout': rng})
        return focal_loss(logits, y, class_weights)

    loss, grads = jax.value_and_grad(loss_fn)(state.params)
    state = state.apply_gradients(grads=grads)
    return state, loss


@jit
def train_step_dnn(state, x, y, rng, class_weights):
    def loss_fn(params):
        vars_ = {'params': params, 'batch_stats': state.batch_stats}
        logits, new_state = state.apply_fn(vars_, x, training=True, rngs={'dropout': rng}, mutable=['batch_stats'])
        loss = focal_loss(logits, y, class_weights)
        return loss, new_state

    (loss, new_state), grads = jax.value_and_grad(loss_fn, has_aux=True)(state.params)
    state = state.apply_gradients(grads=grads)
    state = state.replace(batch_stats=new_state['batch_stats'])
    return state, loss


@jit
def train_step_fse(state, x, mask, y, rng, class_weights):
    def loss_fn(params):
        logits = state.apply_fn({'params': params}, x, mask, training=True, rngs={'dropout': rng})
        return focal_loss(logits, y, class_weights)

    loss, grads = jax.value_and_grad(loss_fn)(state.params)
    state = state.apply_gradients(grads=grads)
    return state, loss


@jit
def train_step_fse_aware(state, x, mask, modes, y, rng, class_weights):
    def loss_fn(params):
        logits = state.apply_fn({'params': params}, x, mask, modes, training=True, rngs={'dropout': rng})
        return focal_loss(logits, y, class_weights)

    loss, grads = jax.value_and_grad(loss_fn)(state.params)
    state = state.apply_gradients(grads=grads)
    return state, loss


@jit
def eval_step_simplenn(state, x, y):
    logits = state.apply_fn({'params': state.params}, x, training=False)
    pred = jnp.argmax(logits, axis=-1)
    acc = jnp.mean(pred == y)
    return acc, logits


@jit
def eval_step_dnn(state, x, y):
    vars_ = {'params': state.params, 'batch_stats': state.batch_stats}
    logits = state.apply_fn(vars_, x, training=False)
    pred = jnp.argmax(logits, axis=-1)
    acc = jnp.mean(pred == y)
    return acc, logits


@jit
def eval_step_fse(state, x, mask, y):
    logits = state.apply_fn({'params': state.params}, x, mask, training=False)
    pred = jnp.argmax(logits, axis=-1)
    acc = jnp.mean(pred == y)
    return acc, logits


@jit
def eval_step_fse_aware(state, x, mask, modes, y):
    logits = state.apply_fn({'params': state.params}, x, mask, modes, training=False)
    pred = jnp.argmax(logits, axis=-1)
    acc = jnp.mean(pred == y)
    return acc, logits


def batch_eval_simple(state, X, y, batch_size=1024):
    accs = []
    logits_list = []
    n_batches = (len(X) + batch_size - 1) // batch_size
    for i in range(n_batches):
        s, e = i * batch_size, min((i + 1) * batch_size, len(X))
        bx = jnp.array(X[s:e], dtype=jnp.float32)
        by = jnp.array(y[s:e], dtype=jnp.int32)
        a, l = eval_step_simplenn(state, bx, by)
        accs.append(float(a))
        logits_list.append(np.array(l))  # convert to NumPy to avoid huge JAX array
    logits = np.concatenate(logits_list, axis=0)
    return float(np.mean(accs)), jnp.array(logits, dtype=jnp.float32)


def batch_eval_dnn(state, X, y, batch_size=1024):
    accs = []
    logits_list = []
    n_batches = (len(X) + batch_size - 1) // batch_size
    for i in range(n_batches):
        s, e = i * batch_size, min((i + 1) * batch_size, len(X))
        bx = jnp.array(X[s:e], dtype=jnp.float32)
        by = jnp.array(y[s:e], dtype=jnp.int32)
        a, l = eval_step_dnn(state, bx, by)
        accs.append(float(a))
        logits_list.append(np.array(l))
    logits = np.concatenate(logits_list, axis=0)
    return float(np.mean(accs)), jnp.array(logits, dtype=jnp.float32)


def batch_eval_fse(state, X, M, y, batch_size=1024):
    accs = []
    logits_list = []
    n_batches = (len(X) + batch_size - 1) // batch_size
    for i in range(n_batches):
        s, e = i * batch_size, min((i + 1) * batch_size, len(X))
        bx = jnp.array(X[s:e], dtype=jnp.float32)
        bm = jnp.array(M[s:e], dtype=jnp.float32)
        by = jnp.array(y[s:e], dtype=jnp.int32)
        a, l = eval_step_fse(state, bx, bm, by)
        accs.append(float(a))
        logits_list.append(np.array(l))
    logits = np.concatenate(logits_list, axis=0)
    return float(np.mean(accs)), jnp.array(logits, dtype=jnp.float32)


def batch_eval_fse_aware(state, X, M, modes, y, batch_size=1024):
    accs = []
    logits_list = []
    n_batches = (len(X) + batch_size - 1) // batch_size
    for i in range(n_batches):
        s, e = i * batch_size, min((i + 1) * batch_size, len(X))
        bx = jnp.array(X[s:e], dtype=jnp.float32)
        bm = jnp.array(M[s:e], dtype=jnp.float32)
        md = jnp.array(modes[s:e], dtype=jnp.int32)
        by = jnp.array(y[s:e], dtype=jnp.int32)
        a, l = eval_step_fse_aware(state, bx, bm, md, by)
        accs.append(float(a))
        logits_list.append(np.array(l))
    logits = np.concatenate(logits_list, axis=0)
    return float(np.mean(accs)), jnp.array(logits, dtype=jnp.float32)


def train_model(model_type, momentum_range, preprocessing_data, force_training=False):
    prep = preprocessing_data  # alias internally

    key = random.PRNGKey(RANDOM_SEED + hash(model_type) % 10000)
    mr_key = momentum_range['key']
    params_cfg = HYPERPARAMETERS[model_type]

    print("\n===", model_type, "|", momentum_range['name'], "===")

    if not force_training:
        loaded, _ = load_single_model(mr_key, model_type)
        if loaded is not None:
            print("Using saved model.")
            return loaded

    # Keep training data in NumPy only
    X_train = prep['X_train']
    X_test = prep['X_test']
    y_train = prep['y_train']
    y_test = prep['y_test']

    masks_train = prep.get('masks_train', None)
    masks_test = prep.get('masks_test', None)
    modes_train = prep.get('detector_modes_train', None)
    modes_test = prep.get('detector_modes_test', None)

    # Class weights (NumPy → small array, OK)
    cw = compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
    class_w = jnp.array(list(dict(enumerate(cw)).values()), dtype=jnp.float32)

    print("Class weights:")
    for i, w in enumerate(cw):
        print(f"  {PARTICLE_NAMES[i]:10s}: {w:.3f}")

    key, init_key = random.split(key)

    # --- Model initialisation ---
    if model_type == 'JAX_SimpleNN':
        model = JAX_SimpleNN(
            hidden_dims=params_cfg['hidden_dims'],
            num_classes=NUM_CLASSES,
            dropout_rate=params_cfg['dropout_rate'],
        )
        dummy = jnp.ones((1, X_train.shape[1]), dtype=jnp.float32)
        vars_ = model.init(init_key, dummy, training=False)
        state = train_state.TrainState.create(
            apply_fn=model.apply,
            params=vars_['params'],
            tx=optax.adam(params_cfg['learning_rate']),
        )
        train_fn = train_step_simplenn
        eval_fn = batch_eval_simple

    elif model_type == 'JAX_DNN':
        model = JAX_DNN(
            hidden_dims=params_cfg['hidden_dims'],
            num_classes=NUM_CLASSES,
            dropout_rate=params_cfg['dropout_rate'],
        )
        dummy = jnp.ones((1, X_train.shape[1]), dtype=jnp.float32)
        vars_ = model.init(init_key, dummy, training=True)
        state = TrainStateWithBatchStats.create(
            apply_fn=model.apply,
            params=vars_['params'],
            tx=optax.adam(params_cfg['learning_rate']),
            batch_stats=vars_.get('batch_stats', {}),
        )
        train_fn = train_step_dnn
        eval_fn = batch_eval_dnn

    elif model_type == 'JAX_FSE_Attention':
        model = JAX_FSE_Attention(
            hidden_dim=params_cfg['hidden_dim'],
            num_heads=params_cfg['num_heads'],
            num_classes=NUM_CLASSES,
            dropout_rate=params_cfg['dropout_rate'],
        )
        dummy_x = jnp.ones((1, X_train.shape[1]), dtype=jnp.float32)
        dummy_m = jnp.ones((1, masks_train.shape[1]), dtype=jnp.float32)
        vars_ = model.init(init_key, dummy_x, dummy_m, training=False)
        state = train_state.TrainState.create(
            apply_fn=model.apply,
            params=vars_['params'],
            tx=optax.adam(params_cfg['learning_rate']),
        )
        train_fn = train_step_fse
        eval_fn = batch_eval_fse

    else:  # Detector-aware FSE
        model = JAX_FSE_Attention_DetectorAware(
            hidden_dim=params_cfg['hidden_dim'],
            num_heads=params_cfg['num_heads'],
            num_classes=NUM_CLASSES,
            dropout_rate=params_cfg['dropout_rate'],
            detector_embed_dim=params_cfg['detector_embed_dim'],
        )
        dummy_x = jnp.ones((1, X_train.shape[1]), dtype=jnp.float32)
        dummy_m = jnp.ones((1, masks_train.shape[1]), dtype=jnp.float32)
        dummy_mode = jnp.zeros((1,), dtype=jnp.int32)
        vars_ = model.init(init_key, dummy_x, dummy_m, dummy_mode, training=False)
        state = train_state.TrainState.create(
            apply_fn=model.apply,
            params=vars_['params'],
            tx=optax.adam(params_cfg['learning_rate']),
        )
        train_fn = train_step_fse_aware
        eval_fn = batch_eval_fse_aware

    # --- Training loop ---
    num_batches = len(X_train) // params_cfg['batch_size']
    best_val = 0.0
    best_params = state.params
    best_batch_stats = getattr(state, 'batch_stats', None)
    patience_counter = 0
    train_losses, val_accs = [], []

    for epoch in range(params_cfg['num_epochs']):
        key, shuf_key, drop_key = random.split(key, 3)
        perm = np.random.permutation(len(X_train))

        epoch_losses = []
        for b in range(num_batches):
            s = b * params_cfg['batch_size']
            e = s + params_cfg['batch_size']

            # --- Batch conversion to JAX ---
            bx = jnp.array(X_train[perm[s:e]], dtype=jnp.float32)
            by = jnp.array(y_train[perm[s:e]], dtype=jnp.int32)

            if masks_train is not None:
                bm = jnp.array(masks_train[perm[s:e]], dtype=jnp.float32)
            if modes_train is not None:
                md = jnp.array(modes_train[perm[s:e]], dtype=jnp.int32)

            drop_key, subkey = random.split(drop_key)
            if model_type == 'JAX_FSE_Attention':
                state, loss = train_fn(state, bx, bm, by, subkey, class_w)
            elif model_type == 'JAX_FSE_Attention_DetectorAware':
                state, loss = train_fn(state, bx, bm, md, by, subkey, class_w)
            else:
                state, loss = train_fn(state, bx, by, subkey, class_w)

            epoch_losses.append(float(loss))

        avg_loss = float(np.mean(epoch_losses))
        train_losses.append(avg_loss)

        # --- Batch-wise evaluation ---
        if model_type == 'JAX_FSE_Attention':
            val_acc, _ = eval_fn(state, X_test, masks_test, y_test, batch_size=1024)
        elif model_type == 'JAX_FSE_Attention_DetectorAware':
            val_acc, _ = eval_fn(state, X_test, masks_test, modes_test, y_test, batch_size=1024)
        else:
            val_acc, _ = eval_fn(state, X_test, y_test, batch_size=1024)

        val_acc = float(val_acc)
        val_accs.append(val_acc)

        if (epoch + 1) % 10 == 0 or epoch == 0:
            print(f"Epoch {epoch+1:3d}: loss={avg_loss:.4f}, val_acc={val_acc:.4f}")

        if val_acc > best_val:
            best_val = val_acc
            best_params = state.params
            if hasattr(state, 'batch_stats'):
                best_batch_stats = state.batch_stats
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= params_cfg['patience']:
                print(f"Early stop at epoch {epoch+1}.")
                break

    state = state.replace(params=best_params)
    if isinstance(state, TrainStateWithBatchStats) and best_batch_stats is not None:
        state = state.replace(batch_stats=best_batch_stats)

    # --- Batch-wise final evaluation ---
    if model_type == 'JAX_FSE_Attention':
        train_acc, train_logits = eval_fn(state, X_train, masks_train, y_train, batch_size=1024)
        test_acc, test_logits = eval_fn(state, X_test, masks_test, y_test, batch_size=1024)
    elif model_type == 'JAX_FSE_Attention_DetectorAware':
        train_acc, train_logits = eval_fn(state, X_train, masks_train, modes_train, y_train, batch_size=1024)
        test_acc, test_logits = eval_fn(state, X_test, masks_test, modes_test, y_test, batch_size=1024)
    else:
        train_acc, train_logits = eval_fn(state, X_train, y_train, batch_size=1024)
        test_acc, test_logits = eval_fn(state, X_test, y_test, batch_size=1024)

    # Convert logits → probabilities in batches to save memory
    train_probs = jax.nn.softmax(train_logits, axis=-1)
    test_probs = jax.nn.softmax(test_logits, axis=-1)
    y_pred_test = jnp.argmax(test_logits, axis=-1)

    print(f"Train acc: {float(train_acc):.4f}, Test acc: {float(test_acc):.4f}, Best val: {best_val:.4f}")

    # --- Free large arrays ---
    del X_train, X_test, y_train, y_test
    if masks_train is not None: del masks_train, masks_test
    if modes_train is not None: del modes_train, modes_test
    gc.collect()

    results = {
        'model_type': model_type,
        'hyperparameters': params_cfg,
        'train_losses': train_losses,
        'val_accuracies': val_accs,
        'best_val_acc': float(best_val),
        'train_acc': float(train_acc),
        'test_acc': float(test_acc),
        'train_probs': train_probs,
        'test_probs': test_probs,
        'y_pred_test': y_pred_test,
        'y_test': y_test,
    }

    save_single_model(mr_key, model_type, results)
    return results


## Section 4: Model Save / Load

In [ ]:
def get_model_path(momentum_range_key, model_type, mode="save"):
    sub = "trained_models"
    work = f"/kaggle/working/{sub}/{momentum_range_key}_{model_type}.pkl"
    inp = f"/kaggle/input/jax-models/jax-models/{momentum_range_key}_{model_type}.pkl"
    return work if mode == "save" else (work if os.path.exists(work) else inp)


def load_single_model(momentum_range_key, model_type):
    path = get_model_path(momentum_range_key, model_type, mode="load")
    if os.path.exists(path):
        with open(path, "rb") as f:
            results = pickle.load(f)
        print(f"Loaded: {os.path.basename(path)}")
        return results, path
    return None, path


def save_single_model(momentum_range_key, model_type, results):
    path = get_model_path(momentum_range_key, model_type, mode="save")
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "wb") as f:
        pickle.dump(results, f)
    print(f"Saved: {os.path.basename(path)}")


### Section 4.0: Data Loading & Initialisation

In [ ]:
print("\n[4.0] Loading data and initialising result storage...")

df = load_all_csv(CSV_PATTERN)      # or load_all_csv, use your existing loader

# Master container: one entry per momentum range
all_results_by_model_and_range = {}

for momentum_range in MOMENTUM_RANGES:
    mr_key = momentum_range['key']
    all_results_by_model_and_range[mr_key] = {
        "preprocessing": None,
        "models": {}
    }

print("Data loaded")

def ensure_preprocessing(mr_key, momentum_range):
    """Compute preprocessing once per momentum range and reuse across models."""
    entry = all_results_by_model_and_range[mr_key]
    if entry["preprocessing"] is None:
        print(f"  Preprocessing data for {momentum_range['name']} ...")
        prep = preprocess_momentum_range(df, momentum_range)
        entry["preprocessing"] = prep
    return entry["preprocessing"]


### Section 4.1: SimpleNN Training

In [ ]:
print("\n" + "="*70)
print("SECTION 4.1 – TRAINING JAX_SimpleNN")
print("="*70)

for momentum_range in MOMENTUM_RANGES:
    mr_key = momentum_range['key']
    mr_name = momentum_range['name']

    print("\n" + "-"*60)
    print(f"[SimpleNN] MOMENTUM RANGE: {mr_name}")
    print("-"*60)

    # 1) Ensure preprocessing exists
    prep = ensure_preprocessing(mr_key, momentum_range)

    # 2) Decide if we force training
    force = FORCE_TRAINING["JAX_SimpleNN"][mr_key]

    # 3) Call unified trainer only for SimpleNN
    results = train_model(
        model_type="JAX_SimpleNN",
        momentum_range=momentum_range,
        preprocessing_data=prep,
        force_training=force,
    )

    # 4) Store in master dict
    all_results_by_model_and_range[mr_key]["models"]["JAX_SimpleNN"] = results

print("\n[4.1] JAX_SimpleNN training / loading complete for all ranges.")


### Section 4.2: DNN (+ BatchNorm) Training

In [ ]:
print("\n" + "="*70)
print("SECTION 4.2 – TRAINING JAX_DNN")
print("="*70)

for momentum_range in MOMENTUM_RANGES:
    mr_key = momentum_range['key']
    mr_name = momentum_range['name']

    print("\n" + "-"*60)
    print(f"[DNN] MOMENTUM RANGE: {mr_name}")
    print("-"*60)

    prep = ensure_preprocessing(mr_key, momentum_range)
    force = FORCE_TRAINING["JAX_DNN"][mr_key]

    results = train_model(
        model_type="JAX_DNN",
        momentum_range=momentum_range,
        preprocessing_data=prep,
        force_training=force,
    )

    all_results_by_model_and_range[mr_key]["models"]["JAX_DNN"] = results

print("\n[4.2] JAX_DNN training / loading complete for all ranges.")


### Section 4.3: FSE + Attention Training (Phase 0)

In [ ]:
print("\n" + "="*70)
print("SECTION 4.3 – TRAINING JAX_FSE_Attention (Phase 0)")
print("="*70)

for momentum_range in MOMENTUM_RANGES:
    mr_key = momentum_range['key']
    mr_name = momentum_range['name']

    print("\n" + "-"*60)
    print(f"[FSE] MOMENTUM RANGE: {mr_name}")
    print("-"*60)

    prep = ensure_preprocessing(mr_key, momentum_range)
    force = FORCE_TRAINING["JAX_FSE_Attention"][mr_key]

    results = train_model(
        model_type="JAX_FSE_Attention",
        momentum_range=momentum_range,
        preprocessing_data=prep,
        force_training=force,
    )

    all_results_by_model_and_range[mr_key]["models"]["JAX_FSE_Attention"] = results

print("\n[4.3] JAX_FSE_Attention (Phase 0) training / loading complete.")


### Section 4.3: FSE + Attention Detector-Aware Training (Phase 1)

In [ ]:
print("\n" + "="*70)
print("SECTION 4.4 – TRAINING JAX_FSE_Attention_DetectorAware (Phase 1)")
print("="*70)

for momentum_range in MOMENTUM_RANGES:
    mr_key = momentum_range['key']
    mr_name = momentum_range['name']

    print("\n" + "-"*60)
    print(f"[FSE Detector-Aware] MOMENTUM RANGE: {mr_name}")
    print("-"*60)

    prep = ensure_preprocessing(mr_key, momentum_range)
    force = FORCE_TRAINING["JAX_FSE_Attention_DetectorAware"][mr_key]

    results = train_model(
        model_type="JAX_FSE_Attention_DetectorAware",
        momentum_range=momentum_range,
        preprocessing_data=prep,
        force_training=force,
    )

    all_results_by_model_and_range[mr_key]["models"]["JAX_FSE_Attention_DetectorAware"] = results

print("\n[4.4] Detector-Aware FSE (Phase 1) training / loading complete.")


## Section 5: Main Orchestration & Simple Comparison

In [ ]:
print("\nLoading full dataset from CSV pattern...")
df = load_all_csv(CSV_PATTERN)

all_results = {}

for mr in MOMENTUM_RANGES:
    key = mr['key']
    prep = preprocess_momentum_range(df, mr)
    all_results[key] = {'preprocessing': prep, 'models': {}}

    for mt in MODEL_TYPES:
        res = train_model(mt, mr, prep, force_training=FORCE_TRAINING[mt][key])
        all_results[key]['models'][mt] = res

all_results_by_model_and_range = {}
for mr in MOMENTUM_RANGES:
    key = mr['key']
    all_results_by_model_and_range[key] = {
        'preprocessing': all_results[key]['preprocessing'],
        'models': all_results[key]['models'],
    }

print("\nSimple accuracy summary (Detector-aware FSE should be best in most ranges):")
for mr in MOMENTUM_RANGES:
    key = mr['key']
    print("\n", mr['name'])
    for mt in MODEL_TYPES:
        acc = all_results[key]['models'][mt]['test_acc']
        print(f"  {mt:35s}: {acc:.4f}")

# Confusion matrices for best model in each range (numeric summary only)
print("\nBest model per momentum range (by test accuracy):")
for mr in MOMENTUM_RANGES:
    key = mr['key']
    best_acc = -1.0
    best_mt = None
    best_res = None
    for mt in MODEL_TYPES:
        res = all_results[key]['models'][mt]
        if res['test_acc'] > best_acc:
            best_acc = res['test_acc']
            best_mt = mt
            best_res = res
    print(f"  {mr['name']}: {best_mt} (test_acc={best_acc:.4f})")

    y_true = np.array(best_res['y_test'])
    y_pred = np.array(best_res['y_pred_test'])
    cm = confusion_matrix(y_true, y_pred, labels=list(range(NUM_CLASSES)))
    print("    Confusion matrix (rows=true, cols=pred):")
    print(cm)

## Section 6: Metrics & Plots

In [ ]:
sns.set_style('whitegrid')

# --- Small utilities -------------------------------------------------------

def get_range_name(mr_key):
    for mr in MOMENTUM_RANGES:
        if mr['key'] == mr_key:
            return mr['name']
    return mr_key


def get_model_results(mr_key, model_type):
    mr_data = all_results_by_model_and_range[mr_key]
    return mr_data['models'][model_type]


def get_preprocessing(mr_key):
    return all_results_by_model_and_range[mr_key]['preprocessing']


# ---------------------------------------------------------------------------
# 6.1 Test Accuracy Comparison for All Models (per momentum range)
# ---------------------------------------------------------------------------

fig, axes = plt.subplots(1, len(MOMENTUM_RANGES), figsize=(6 * len(MOMENTUM_RANGES), 5))
if len(MOMENTUM_RANGES) == 1:
    axes = [axes]

for plot_idx, (mr_key, mr_data) in enumerate(all_results_by_model_and_range.items()):
    ax = axes[plot_idx]
    mr_name = get_range_name(mr_key)

    model_labels = []
    test_accs = []
    colors = []

    if 'models' in mr_data:
        for model_type in MODEL_TYPES:
            if model_type in mr_data['models']:
                model_labels.append(model_type.replace('JAX_', ''))
                test_accs.append(mr_data['models'][model_type]['test_acc'])
                colors.append(model_colors.get(model_type, '#555555'))

    x = np.arange(len(model_labels))
    width = 0.6

    bars = ax.bar(x, test_accs, width, color=colors, alpha=0.85, edgecolor='black', linewidth=1.3)
    for bar, val in zip(bars, test_accs):
        ax.text(bar.get_x() + bar.get_width() / 2.0, val + 0.005,
                f"{val:.4f}", ha='center', va='bottom', fontsize=9, fontweight='bold')

    ax.set_xlabel('Model Type', fontsize=11, fontweight='bold')
    ax.set_ylabel('Test Accuracy', fontsize=11, fontweight='bold')
    ax.set_title(mr_name, fontsize=12, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels(model_labels, rotation=45, ha='right', fontsize=9)
    ax.set_ylim(0.4, 1.0)
    ax.grid(axis='y', alpha=0.3)

plt.suptitle('Test Accuracy – All Models per Momentum Range', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()


# ---------------------------------------------------------------------------
# 6.2 Confusion Matrices – Best Model per Range
# ---------------------------------------------------------------------------

fig, axes = plt.subplots(1, len(MOMENTUM_RANGES), figsize=(6 * len(MOMENTUM_RANGES), 5))
if len(MOMENTUM_RANGES) == 1:
    axes = [axes]

for plot_idx, (mr_key, mr_data) in enumerate(all_results_by_model_and_range.items()):
    ax = axes[plot_idx]
    mr_name = get_range_name(mr_key)

    best_acc = -1.0
    best_model_type = None
    best_results = None

    if 'models' in mr_data:
        for model_type in MODEL_TYPES:
            if model_type in mr_data['models']:
                res = mr_data['models'][model_type]
                if res['test_acc'] > best_acc:
                    best_acc = res['test_acc']
                    best_model_type = model_type
                    best_results = res

    if best_results is None:
        ax.text(0.5, 0.5, 'No model', ha='center', va='center', transform=ax.transAxes)
        continue

    y_true = np.array(best_results['y_test'])
    y_pred = np.array(best_results['y_pred_test'])
    cm = confusion_matrix(y_true, y_pred, labels=list(range(NUM_CLASSES)))
    cm_norm = cm.astype('float') / cm.sum(axis=1, keepdims=True)

    sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Blues',
                xticklabels=PARTICLE_NAMES, yticklabels=PARTICLE_NAMES,
                cbar_kws={'shrink': 0.8}, ax=ax)

    ax.set_xlabel('Predicted', fontsize=10, fontweight='bold')
    ax.set_ylabel('True', fontsize=10, fontweight='bold')
    ax.set_title(f"{mr_name}\n{best_model_type.replace('JAX_', '')} (Acc={best_acc:.4f})",
                 fontsize=11, fontweight='bold')

plt.suptitle('Confusion Matrices – Best Model per Momentum Range', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()


# ---------------------------------------------------------------------------
# 6.3 Model Performance Ranking – Grouped by Momentum Range
# ---------------------------------------------------------------------------

perf_data = []
labels_full = []
colors_list = []

for mr_key, mr_data in all_results_by_model_and_range.items():
    mr_name = get_range_name(mr_key).split('(')[0].strip()

    if 'models' not in mr_data:
        continue

    range_models = []
    for model_type in MODEL_TYPES:
        if model_type in mr_data['models']:
            res = mr_data['models'][model_type]
            range_models.append({
                'model_type': model_type.replace('JAX_', ''),
                'test_acc': res['test_acc'],
                'mr_name': mr_name,
            })

    range_models_sorted = sorted(range_models, key=lambda x: x['test_acc'], reverse=True)

    for m in range_models_sorted:
        perf_data.append(m['test_acc'])
        labels_full.append(f"{m['mr_name']} – {m['model_type']}")
        colors_list.append(model_colors.get('JAX_' + m['model_type'], '#555555'))

fig, ax = plt.subplots(figsize=14, 8)

y_pos = np.arange(len(perf_data))

bars = ax.barh(y_pos, perf_data, color=colors_list, alpha=0.85, edgecolor='black', linewidth=1.3)

for bar, val in zip(bars, perf_data):
    ax.text(val + 0.005, bar.get_y() + bar.get_height() / 2.0,
            f"{val:.4f}", va='center', fontsize=9, fontweight='bold')

ax.set_ylabel('Model – Momentum Range', fontsize=11, fontweight='bold')
ax.set_xlabel('Test Accuracy', fontsize=11, fontweight='bold')
ax.set_yticks(y_pos)
ax.set_yticklabels(labels_full, fontsize=9)
ax.set_xlim(0.4, 1.05)
ax.grid(axis='x', alpha=0.3)
ax.set_title('Model Performance Ranking – Grouped by Momentum Range', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.show()


# ---------------------------------------------------------------------------
# 6.4 One-vs-Rest ROC/AUC Curves – All Models, All Ranges (Per-Particle)
# ---------------------------------------------------------------------------

fig, axes = plt.subplots(len(MOMENTUM_RANGES), NUM_CLASSES,
                         figsize=(4 * NUM_CLASSES, 3.5 * len(MOMENTUM_RANGES)),
                         sharex=True, sharey=True)

if len(MOMENTUM_RANGES) == 1:
    axes = np.array([axes])

for row_idx, (mr_key, mr_data) in enumerate(all_results_by_model_and_range.items()):
    mr_name = get_range_name(mr_key)

    for class_idx, particle in enumerate(PARTICLE_NAMES):
        ax = axes[row_idx, class_idx]
        for model_type in MODEL_TYPES:
            if 'models' not in mr_data or model_type not in mr_data['models']:
                continue
            res = mr_data['models'][model_type]
            y_true = np.array(res['y_test'])
            y_prob = np.array(res['test_probs'])[:, class_idx]

            y_bin = (y_true == class_idx).astype(int)
            if np.unique(y_bin).size < 2:
                continue

            fpr, tpr, _ = roc_curve(y_bin, y_prob)
            roc_auc = auc(fpr, tpr)
            label = f"{model_type.replace('JAX_', '')} (AUC={roc_auc:.3f})"
            ax.plot(fpr, tpr, label=label,
                    color=model_colors.get(model_type, None), linewidth=1.5, alpha=0.9)

        ax.plot([0, 1], [0, 1], 'k--', linewidth=1, alpha=0.3)
        if row_idx == len(MOMENTUM_RANGES) - 1:
            ax.set_xlabel('False Positive Rate', fontsize=9)
        if class_idx == 0:
            ax.set_ylabel('True Positive Rate', fontsize=9)
        ax.set_title(f"{mr_name}\n{particle}", fontsize=9)
        ax.grid(alpha=0.3)
        if row_idx == 0 and class_idx == NUM_CLASSES - 1:
            ax.legend(fontsize=7, loc='lower right')

plt.suptitle('One-vs-Rest ROC Curves – All Models, All Ranges (Per Particle)',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()


# ---------------------------------------------------------------------------
# 6.5 Macro AUC by Metric – Efficiency, Purity, F1-Score (All Models & Ranges)
# ---------------------------------------------------------------------------

comparison_metrics = {
    'efficiency_auc': [],
    'purity_auc': [],
    'f1_auc': [],
}
model_labels = []
range_labels = []

for momentum_range in MOMENTUM_RANGES:
    mr_key = momentum_range['key']
    mr_name = momentum_range['name']
    mr_data = all_results_by_model_and_range[mr_key]

    if 'models' not in mr_data:
        continue

    for model_type in MODEL_TYPES:
        if model_type not in mr_data['models']:
            continue

        res = mr_data['models'][model_type]
        y_test = np.array(res['y_test'])
        y_prob = np.array(res['test_probs'])

        y_bin = label_binarize(y_test, classes=np.arange(NUM_CLASSES))

        eff_aucs = []
        pur_aucs = []
        f1_aucs = []

        for i in range(NUM_CLASSES):
            if np.unique(y_bin[:, i]).size < 2:
                continue

            fpr, tpr, _ = roc_curve(y_bin[:, i], y_prob[:, i])
            roc_auc = auc(fpr, tpr)
            eff_aucs.append(roc_auc)

            prec, rec, _ = precision_recall_curve(y_bin[:, i], y_prob[:, i])
            pr_auc = auc(rec, prec)
            pur_aucs.append(pr_auc)

            f1_proxy = 0.5 * (roc_auc + pr_auc)
            f1_aucs.append(f1_proxy)

        if not eff_aucs:
            continue

        comparison_metrics['efficiency_auc'].append(np.mean(eff_aucs))
        comparison_metrics['purity_auc'].append(np.mean(pur_aucs))
        comparison_metrics['f1_auc'].append(np.mean(f1_aucs))

        model_labels.append(model_type.replace('JAX_', ''))
        range_labels.append(mr_name)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
metric_keys = ['efficiency_auc', 'purity_auc', 'f1_auc']
metric_titles = ['Efficiency ROC AUC', 'Purity PR AUC', 'F1-Score Macro AUC']

for idx, (metric_key, title) in enumerate(zip(metric_keys, metric_titles)):
    ax = axes[idx]

    positions = []
    values = []
    colors = []

    pos = 0.0
    for mr in MOMENTUM_RANGES:
        mr_name = mr['name']

        data_for_range = []
        colors_for_range = []
        for i, (m_label, r_label) in enumerate(zip(model_labels, range_labels)):
            if r_label == mr_name:
                data_for_range.append(comparison_metrics[metric_key][i])
                colors_for_range.append(model_colors.get('JAX_' + m_label, '#555555'))

        for v, c in zip(data_for_range, colors_for_range):
            positions.append(pos)
            values.append(v)
            colors.append(c)
            pos += 1.0
        pos += 0.5  # spacing between ranges

    bars = ax.bar(positions, values, color=colors, alpha=0.85,
                  edgecolor='black', linewidth=1.3, width=0.9)

    for bar, val in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width() / 2.0, val + 0.01,
                f"{val:.3f}", ha='center', va='bottom', fontsize=8, fontweight='bold')

    ax.set_ylabel('Macro AUC', fontsize=11, fontweight='bold')
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_ylim(0.5, 1.05)
    ax.grid(axis='y', alpha=0.3)
    ax.set_xticks([])

plt.suptitle('Model Performance – Macro AUC by Metric (All Models, All Ranges)',
             fontsize=14, fontweight='bold', y=0.98)
plt.tight_layout(rect=[0, 0.05, 1, 0.96])
plt.show()


# ---------------------------------------------------------------------------
# 6.6 Phase 1: Standard FSE vs Detector-Aware FSE (by Detector Mode)
# ---------------------------------------------------------------------------

comparison_results = []  # reused later for TPC-only improvement plot

mode_names = {0: 'NONE', 1: 'TPC_ONLY', 2: 'TOF_ONLY', 3: 'TPC_TOF'}

for momentum_range in MOMENTUM_RANGES:
    mr_key = momentum_range['key']
    mr_name = momentum_range['name']
    mr_data = all_results_by_model_and_range[mr_key]

    if 'models' not in mr_data:
        continue
    if ('JAX_FSE_Attention' not in mr_data['models'] or
        'JAX_FSE_Attention_DetectorAware' not in mr_data['models']):
        continue

    prep = mr_data['preprocessing']
    y_test = np.array(prep['y_test'])
    modes_test = np.array(prep['detector_modes_test'])

    std_res = mr_data['models']['JAX_FSE_Attention']
    aware_res = mr_data['models']['JAX_FSE_Attention_DetectorAware']
    y_pred_std = np.array(std_res['y_pred_test'])
    y_pred_aware = np.array(aware_res['y_pred_test'])

    print(f"\n{mr_name}")
    acc_std = accuracy_score(y_test, y_pred_std)
    acc_aware = accuracy_score(y_test, y_pred_aware)
    print(f"  Overall Standard FSE:       {acc_std:.4f}")
    print(f"  Overall Detector-Aware FSE: {acc_aware:.4f}  (Δ={acc_aware - acc_std:.4f})")

    for mode_id in sorted(mode_names.keys()):
        mask = (modes_test == mode_id)
        if mask.sum() == 0:
            continue
        y_mode = y_test[mask]
        y_std_mode = y_pred_std[mask]
        y_aware_mode = y_pred_aware[mask]

        acc_std_mode = accuracy_score(y_mode, y_std_mode)
        acc_aware_mode = accuracy_score(y_mode, y_aware_mode)
        delta = acc_aware_mode - acc_std_mode

        comparison_results.append({
            'Momentum Range': mr_name,
            'Detector Mode': mode_names[mode_id],
            'Tracks': int(mask.sum()),
            'Std FSE': acc_std_mode,
            'Aware FSE': acc_aware_mode,
            'Delta': delta,
        })

fig, axes = plt.subplots(1, len(MOMENTUM_RANGES), figsize=(6 * len(MOMENTUM_RANGES), 4))
if len(MOMENTUM_RANGES) == 1:
    axes = [axes]

for ax, momentum_range in zip(axes, MOMENTUM_RANGES):
    mr_name = momentum_range['name']
    data_for_range = [d for d in comparison_results if d['Momentum Range'] == mr_name]

    if not data_for_range:
        ax.text(0.5, 0.5, 'No FSE data', ha='center', va='center', transform=ax.transAxes)
        ax.set_title(mr_name)
        continue

    modes = [d['Detector Mode'] for d in data_for_range]
    std_accs = [d['Std FSE'] for d in data_for_range]
    aware_accs = [d['Aware FSE'] for d in data_for_range]

    x = np.arange(len(modes))
    width = 0.35

    bars1 = ax.bar(x - width / 2, std_accs, width, label='Standard FSE',
                   color=model_colors['JAX_FSE_Attention'], alpha=0.8,
                   edgecolor='black', linewidth=1.3)
    bars2 = ax.bar(x + width / 2, aware_accs, width, label='Detector-Aware FSE',
                   color=model_colors['JAX_FSE_Attention_DetectorAware'], alpha=0.8,
                   edgecolor='black', linewidth=1.3)

    for bars in (bars1, bars2):
        for bar in bars:
            h = bar.get_height()
            ax.text(bar.get_x() + bar.get_width() / 2.0, h + 0.005,
                    f"{h:.3f}", ha='center', va='bottom', fontsize=8, fontweight='bold')

    ax.set_ylabel('Accuracy', fontsize=11, fontweight='bold')
    ax.set_title(mr_name, fontsize=12, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels(modes, rotation=45, ha='right', fontsize=9)
    ax.set_ylim(0.4, 1.05)
    ax.grid(axis='y', alpha=0.3)

axes[0].legend(fontsize=9, loc='lower left')
plt.suptitle('Phase 1 – Standard FSE vs Detector-Aware FSE by Detector Mode',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()


# ---------------------------------------------------------------------------
# 6.7 Accuracy Comparison – FSE vs Detector-Aware vs Bayesian PID
# ---------------------------------------------------------------------------

fig, axes = plt.subplots(1, len(MOMENTUM_RANGES), figsize=(6 * len(MOMENTUM_RANGES), 4))
if len(MOMENTUM_RANGES) == 1:
    axes = [axes]

for ax, momentum_range in zip(axes, MOMENTUM_RANGES):
    mr_key = momentum_range['key']
    mr_name = momentum_range['name']
    mr_data = all_results_by_model_and_range[mr_key]

    if 'models' not in mr_data:
        ax.text(0.5, 0.5, 'No models', ha='center', va='center', transform=ax.transAxes)
        continue

    has_fse = 'JAX_FSE_Attention' in mr_data['models']
    has_fse_aware = 'JAX_FSE_Attention_DetectorAware' in mr_data['models']

    prep = mr_data['preprocessing']
    y_test = np.array(prep['y_test'])
    bayes_mask = np.array(prep['bayes_availability_test']).astype(bool)
    bayes_pred = np.array(prep['bayes_pred_original_test'])

    acc_bayes_all = accuracy_score(y_test, bayes_pred)
    if bayes_mask.sum() > 0:
        acc_bayes_real = accuracy_score(y_test[bayes_mask], bayes_pred[bayes_mask])
    else:
        acc_bayes_real = np.nan

    categories = ['All Tracks', 'Real Bayesian']
    x = np.arange(len(categories))
    width = 0.25

    bars_all = []

    if has_fse:
        fse_res = mr_data['models']['JAX_FSE_Attention']
        y_pred_fse = np.array(fse_res['y_pred_test'])
        acc_fse_all = accuracy_score(y_test, y_pred_fse)
        acc_fse_real = accuracy_score(y_test[bayes_mask], y_pred_fse[bayes_mask]) if bayes_mask.sum() > 0 else np.nan
        fse_accs = [acc_fse_all, acc_fse_real]
        bars1 = ax.bar(x - width, fse_accs, width,
                       label='FSE+Attention', color=model_colors['JAX_FSE_Attention'],
                       alpha=0.8, edgecolor='black', linewidth=1.3)
        bars_all.append(bars1)

    if has_fse_aware:
        aware_res = mr_data['models']['JAX_FSE_Attention_DetectorAware']
        y_pred_aware = np.array(aware_res['y_pred_test'])
        acc_aware_all = accuracy_score(y_test, y_pred_aware)
        acc_aware_real = accuracy_score(y_test[bayes_mask], y_pred_aware[bayes_mask]) if bayes_mask.sum() > 0 else np.nan
        aware_accs = [acc_aware_all, acc_aware_real]
        bars2 = ax.bar(x, aware_accs, width,
                       label='FSE Detector-Aware', color=model_colors['JAX_FSE_Attention_DetectorAware'],
                       alpha=0.8, edgecolor='black', linewidth=1.3)
        bars_all.append(bars2)

    bayes_accs = [acc_bayes_all, acc_bayes_real]
    bars3 = ax.bar(x + width, bayes_accs, width,
                   label='Bayesian PID', color=model_colors['Bayesian_PID'],
                   alpha=0.8, edgecolor='black', linewidth=1.3)
    bars_all.append(bars3)

    for bars in bars_all:
        for bar in bars:
            h = bar.get_height()
            if np.isnan(h):
                continue
            ax.text(bar.get_x() + bar.get_width() / 2.0, h + 0.01,
                    f"{h:.3f}", ha='center', va='bottom', fontsize=8, fontweight='bold')

    ax.set_ylabel('Accuracy', fontsize=11, fontweight='bold')
    ax.set_title(mr_name, fontsize=12, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels(categories, fontsize=10)
    ax.set_ylim(0.4, 1.05)
    ax.grid(axis='y', alpha=0.3)

axes[0].legend(fontsize=9, loc='lower left')
plt.suptitle('Accuracy Comparison – FSE+Attention vs Detector-Aware vs Bayesian PID',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()


# ---------------------------------------------------------------------------
# 6.8 Per-Particle Accuracy – FSE vs Detector-Aware vs Bayesian PID
# ---------------------------------------------------------------------------

fig, axes = plt.subplots(1, len(MOMENTUM_RANGES), figsize=(6 * len(MOMENTUM_RANGES), 4))
if len(MOMENTUM_RANGES) == 1:
    axes = [axes]

for ax, momentum_range in zip(axes, MOMENTUM_RANGES):
    mr_key = momentum_range['key']
    mr_name = momentum_range['name']
    mr_data = all_results_by_model_and_range[mr_key]

    if 'models' not in mr_data:
        ax.text(0.5, 0.5, 'No models', ha='center', va='center', transform=ax.transAxes)
        continue

    has_fse = 'JAX_FSE_Attention' in mr_data['models']
    has_fse_aware = 'JAX_FSE_Attention_DetectorAware' in mr_data['models']

    prep = mr_data['preprocessing']
    y_test = np.array(prep['y_test'])
    bayes_pred = np.array(prep['bayes_pred_original_test'])

    if has_fse:
        fse_res = mr_data['models']['JAX_FSE_Attention']
        y_pred_fse = np.array(fse_res['y_pred_test'])
    else:
        y_pred_fse = None

    if has_fse_aware:
        aware_res = mr_data['models']['JAX_FSE_Attention_DetectorAware']
        y_pred_aware = np.array(aware_res['y_pred_test'])
    else:
        y_pred_aware = None

    particles = []
    fse_accs = []
    aware_accs = []
    bayes_accs = []

    for i, particle in enumerate(PARTICLE_NAMES):
        mask = (y_test == i)
        if mask.sum() == 0:
            continue

        particles.append(particle)
        if y_pred_fse is not None:
            fse_accs.append(accuracy_score(y_test[mask], y_pred_fse[mask]))
        else:
            fse_accs.append(np.nan)
        if y_pred_aware is not None:
            aware_accs.append(accuracy_score(y_test[mask], y_pred_aware[mask]))
        else:
            aware_accs.append(np.nan)
        bayes_accs.append(accuracy_score(y_test[mask], bayes_pred[mask]))

    x = np.arange(len(particles))
    width = 0.25

    bars_all = []

    bars1 = ax.bar(x - width, fse_accs, width, label='FSE+Attention',
                   color=model_colors['JAX_FSE_Attention'], alpha=0.8,
                   edgecolor='black', linewidth=1.3)
    bars_all.append(bars1)

    bars2 = ax.bar(x, aware_accs, width, label='FSE Detector-Aware',
                   color=model_colors['JAX_FSE_Attention_DetectorAware'], alpha=0.8,
                   edgecolor='black', linewidth=1.3)
    bars_all.append(bars2)

    bars3 = ax.bar(x + width, bayes_accs, width, label='Bayesian PID',
                   color=model_colors['Bayesian_PID'], alpha=0.8,
                   edgecolor='black', linewidth=1.3)
    bars_all.append(bars3)

    for bars in bars_all:
        for bar in bars:
            h = bar.get_height()
            if np.isnan(h):
                continue
            ax.text(bar.get_x() + bar.get_width() / 2.0, h + 0.01,
                    f"{h:.3f}", ha='center', va='bottom', fontsize=8, fontweight='bold')

    ax.set_ylabel('Accuracy', fontsize=11, fontweight='bold')
    ax.set_title(mr_name, fontsize=12, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels(particles, rotation=45, ha='right', fontsize=10)
    ax.set_ylim(0.4, 1.05)
    ax.grid(axis='y', alpha=0.3)

axes[0].legend(fontsize=9, loc='lower left')
plt.suptitle('Per-Particle Accuracy – FSE+Attention vs Detector-Aware vs Bayesian PID',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()


# ---------------------------------------------------------------------------
# 6.9 Improvement of Detector-Aware FSE over Standard FSE – TPC-Only
# ---------------------------------------------------------------------------

# Reuse comparison_results from Section 7.6

ranges = []
improvements_tpc_only = []

for mr in MOMENTUM_RANGES:
    mr_name = mr['name']
    entries = [d for d in comparison_results
               if d['Momentum Range'] == mr_name and d['Detector Mode'] == 'TPC_ONLY']
    if not entries:
        continue
    delta_vals = [d['Delta'] for d in entries]
    ranges.append(mr_name)
    improvements_tpc_only.append(np.mean(delta_vals))

if improvements_tpc_only:
    x = np.arange(len(ranges))
    fig, ax = plt.subplots(figsize=6, 4)
    bars = ax.bar(x, improvements_tpc_only, width=0.6,
                  color=model_colors['JAX_FSE_Attention_DetectorAware'],
                  alpha=0.85, edgecolor='black', linewidth=1.3)

    for bar, val in zip(bars, improvements_tpc_only):
        ax.text(bar.get_x() + bar.get_width() / 2.0,
                val + (0.002 if val >= 0 else -0.002),
                f"{val:+.4f}", ha='center',
                va='bottom' if val >= 0 else 'top',
                fontsize=9, fontweight='bold')

    ax.axhline(0.0, color='k', linestyle='--', linewidth=1.2, alpha=0.7)
    ax.set_ylabel('Accuracy Gain (Aware − Standard)', fontsize=11, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels(ranges, rotation=15, ha='right', fontsize=10)
    ax.set_title('Detector-Aware FSE Improvement over Standard FSE (TPC-Only Tracks)',
                 fontsize=13, fontweight='bold')
    ax.grid(axis='y', alpha=0.3)

    plt.tight_layout()
    plt.show()
else:
    print('No TPC_ONLY entries found in comparison_results; run Section 7.6 first.')
